# Data Acquisition

In this lab, we will explore the process of data acquisition. 

In the case of *passive* network traffic analysis, there are generally two primary ways of acquiring data:
* Packet capture
* Network traffic flows (sometimes called IPFIX)

The advent of more programmability in networks is quickly changing this landscape. 

In particular, systems like Retina are now making it possible to ask more complex questions of network traffic from passive traffic capture and analysis but the general underlying traffic patterns are still based on raw packet capture.

## Background

Because packet captures are so large, it can sometimes be convenient to work with summary statistics about network traffic. Instead of the raw packets, data could represent the total number of bytes, packets, and so forth for flows. 

Raw traffic capture is thus sometimes represented as summaries of flow statistics, rather than raw packet traces. In this activity, we will *generate* the summary statistics and then think about what types of information is (and is not) available in a packet trace summary vs. a raw packet capture.

## Step 1: Load a Packet Trace

Load the packet capture from the last assignment.

In [27]:
import pandas as pd

ndf = pd.read_csv("data/netflix.csv.gz")
ndf.head(20)

,No.,Time,Source,Destination,Protocol,Length,Info
0,1,2018-02-11 08:10:00.534682,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0xed0c A fonts.gstatic.com
1,2,2018-02-11 08:10:00.534832,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,77,Standard query 0x301a AAAA fonts.gstatic.com
2,3,2018-02-11 08:10:00.539408,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x11d3 A googleads.g.doubleclic...
3,4,2018-02-11 08:10:00.541204,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,87,Standard query 0x1284 AAAA googleads.g.doublec...
4,5,2018-02-11 08:10:00.545785,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,78,Standard query 0x3432 AAAA ytimg.l.google.com
5,6,2018-02-11 08:10:00.547036,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,96,Standard query 0xb756 A r4---sn-gxo5uxg-jqbe.g...
6,7,2018-02-11 08:10:00.547156,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x62ab A ssl.gstatic.com
7,8,2018-02-11 08:10:00.547249,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,74,Standard query 0x42fb A www.google.com
8,9,2018-02-11 08:10:00.853950,ns-vip-pro.paris.inria.fr,192.168.43.72,DNS,386,Standard query response 0x11d3 A 216.58.213.162
9,10,2018-02-11 08:10:00.853970,192.168.43.72,ns-vip-pro.paris.inria.fr,DNS,75,Standard query 0x8756 A www.gstatic.com


## Step 2: Generate Statistics for Each Flow

A **flow** is defined as groups of packets that share the following attributes:
* Source IP Address
* Destination IP Address
* Source Port
* Destination Port
* Time interval

The csv file we used in the past assignments do not have port numbers, so you can simply group on source and destination IP address.

For each flow in the packet trace, generate the following statistics for each flow:

* Number of bytes
* Number of packets
* Duration (time)

In [28]:
# Convert times to timestamps if not already so
ndf["Time"] = pd.to_datetime(ndf["Time"])

# Bin by timestamp (to allow for dividing flows into 30s intervals)
# ndf["bin"] = ndf["Time"].dt.floor("30s")

flow_dict = {}

for name, group in ndf.groupby(["Source", "Destination"]):
    # Dict format: flow : 3-tuple of (Total bytes, total packets, total duration)
    flow_dict.update({name: (group["Length"].sum(), len(group["Length"]), group["Time"].iloc[-1] - group["Time"].iloc[0])})

print(flow_dict)

{('0.0.0.0', '255.255.255.255'): (np.int64(3032), 8, Timedelta('0 days 00:07:27.081009')), ('0.0.0.0', 'all-systems.mcast.net'): (np.int64(184), 4, Timedelta('0 days 00:06:16.254753')), ('104.31.113.215', '192.168.43.72'): (np.int64(934), 6, Timedelta('0 days 00:00:02.685329')), ('17.188.166.20', '192.168.43.72'): (np.int64(218), 2, Timedelta('0 days 00:00:07.635267')), ('17.252.44.15', '192.168.43.72'): (np.int64(420), 3, Timedelta('0 days 00:04:17.374504')), ('192.168.1.159', '192.168.43.72'): (np.int64(752), 8, Timedelta('0 days 00:00:00.000827')), ('192.168.43.192', '224.0.0.251'): (np.int64(7072), 31, Timedelta('0 days 00:07:21.345503')), ('192.168.43.37', '224.0.0.251'): (np.int64(882), 10, Timedelta('0 days 00:06:29.531994')), ('192.168.43.72', '104.31.113.215'): (np.int64(814), 8, Timedelta('0 days 00:00:02.698557')), ('192.168.43.72', '17.188.166.20'): (np.int64(442), 3, Timedelta('0 days 00:00:13.982521')), ('192.168.43.72', '17.252.44.15'): (np.int64(574), 5, Timedelta('0 da

### Total Number of Flows

Count the total number of flows in this trace.

In [29]:
print(f"Total flows: {len(flow_dict)}")

Total flows: 77


### Number of Bytes

Count the total number of bytes for each flow in the trace. 

Then, sort the flows by size, in bytes.  

What do you notice about the large flows? What do they look like?

In [30]:
sorted_flows = dict(sorted(flow_dict.items(), key=lambda kv: kv[1][0], reverse=True),)
print(sorted_flows)

{('ipv4-c071-cdg001-ix.1.oca.nflxvideo.net', '192.168.43.72'): (np.int64(120607242), 80084, Timedelta('0 days 00:07:55.340378')), ('ipv4-c069-cdg001-ix.1.oca.nflxvideo.net', '192.168.43.72'): (np.int64(7138148), 4873, Timedelta('0 days 00:07:55.341855')), ('192.168.43.72', 'ipv4-c071-cdg001-ix.1.oca.nflxvideo.net'): (np.int64(3357228), 47902, Timedelta('0 days 00:07:55.478319')), ('a23-57-80-120.deploy.static.akamaitechnologies.com', '192.168.43.72'): (np.int64(1332086), 1005, Timedelta('0 days 00:06:12.896515')), ('ipv4-c063-cdg001-ix.1.oca.nflxvideo.net', '192.168.43.72'): (np.int64(431178), 338, Timedelta('0 days 00:01:15.093257')), ('ec2-52-19-39-146.eu-west-1.compute.amazonaws.com', '192.168.43.72'): (np.int64(348141), 472, Timedelta('0 days 00:07:10.071491')), ('192.168.43.72', 'ec2-52-19-39-146.eu-west-1.compute.amazonaws.com'): (np.int64(340330), 489, Timedelta('0 days 00:07:10.839566')), ('192.168.43.72', 'ipv4-c069-cdg001-ix.1.oca.nflxvideo.net'): (np.int64(244455), 3170, Tim

The largest flows tend to be either from video streaming sites like Netflix or webservers like aws, which might also just be Netflix going through aws. The flows are also generally quite long (like upwards of seven minutes).

### Number of Packets

Count the number of packets in each flow. 

What do you notice about these flows? Are they similar to the largest flows by bytes? Which differ?

In [32]:
sorted_flows = dict(sorted(flow_dict.items(), key=lambda kv: kv[1][1], reverse=True),)
print(sorted_flows)

{('ipv4-c071-cdg001-ix.1.oca.nflxvideo.net', '192.168.43.72'): (np.int64(120607242), 80084, Timedelta('0 days 00:07:55.340378')), ('192.168.43.72', 'ipv4-c071-cdg001-ix.1.oca.nflxvideo.net'): (np.int64(3357228), 47902, Timedelta('0 days 00:07:55.478319')), ('ipv4-c069-cdg001-ix.1.oca.nflxvideo.net', '192.168.43.72'): (np.int64(7138148), 4873, Timedelta('0 days 00:07:55.341855')), ('192.168.43.72', 'ipv4-c069-cdg001-ix.1.oca.nflxvideo.net'): (np.int64(244455), 3170, Timedelta('0 days 00:07:55.487773')), ('a23-57-80-120.deploy.static.akamaitechnologies.com', '192.168.43.72'): (np.int64(1332086), 1005, Timedelta('0 days 00:06:12.896515')), ('192.168.43.72', 'a23-57-80-120.deploy.static.akamaitechnologies.com'): (np.int64(60463), 834, Timedelta('0 days 00:06:13.620743')), ('192.168.43.72', 'ec2-52-19-39-146.eu-west-1.compute.amazonaws.com'): (np.int64(340330), 489, Timedelta('0 days 00:07:10.839566')), ('ec2-52-19-39-146.eu-west-1.compute.amazonaws.com', '192.168.43.72'): (np.int64(348141)

The largest flows by packets tend to be similar to the largest flows by bytes.

### Duration

Compute the duration of each flow, by taking the time of the last packet and subtracting the time of the first, for each flow.  

What are the longest flows in the trace?

In [31]:
sorted_flows = dict(sorted(flow_dict.items(), key=lambda kv: kv[1][2], reverse=True),)
print(sorted_flows)

{('192.168.43.72', 'par10s38-in-f3.1e100.net'): (np.int64(17828), 158, Timedelta('0 days 00:08:15.451688')), ('par10s38-in-f3.1e100.net', '192.168.43.72'): (np.int64(98271), 144, Timedelta('0 days 00:08:15.104425')), ('ns-vip-pro.paris.inria.fr', '192.168.43.72'): (np.int64(20208), 50, Timedelta('0 days 00:08:12.745875')), ('192.168.43.72', 'ns-vip-pro.paris.inria.fr'): (np.int64(4711), 58, Timedelta('0 days 00:08:12.687393')), ('192.168.43.97', '224.0.0.251'): (np.int64(1004), 9, Timedelta('0 days 00:08:06.705466')), ('192.168.43.72', '224.0.0.251'): (np.int64(2528), 16, Timedelta('0 days 00:08:06.400795')), ('fe80::e6ce:8fff:fe01:4c54', 'ff02::fb'): (np.int64(2928), 16, Timedelta('0 days 00:08:06.400733')), ('192.168.43.72', 'ec2-34-252-77-54.eu-west-1.compute.amazonaws.com'): (np.int64(3753), 25, Timedelta('0 days 00:08:00.552744')), ('ec2-34-252-77-54.eu-west-1.compute.amazonaws.com', '192.168.43.72'): (np.int64(4884), 21, Timedelta('0 days 00:08:00.407859')), ('192.168.43.72', 'ip

They're all from 'par10s38-in-f3.1e100.net,' aws, or netflix. 

## Bytes and Packets Per Second

Compute the bytes per second and packets per second for each flow.

For a simple feature computation, compute the average bytes and packets per second for each flow, for the entire duration of the flow.  If you want to get more clever or fancy, you can do "windowed averages", computing bytes or packets per second for shorter time intervals.

## Note

Some of the libraries that we will use in this class, including the `netml` library from the University of Chicago, will compute these and other statistics automatically.

## Thought Questions

1. What are the largest flows in terms of: Number of bytes? Number of packets?

2. What do you notice about the flow sizes and the directions of flows?

3. What kinds of features are *not* available in packet summary statistics like those above which might be available in a raw packet trace? How might those features be useful for different packet classification problems?